# M2: inference-only PO-HRL vs chain-of-thought

Frozen model. The model plans a poset of subgoals, then executes it round by round (all frontier steps in parallel; each executor sees only its prerequisites). Compared with ordinary chain-of-thought on the same problems.

Outputs go to `MyDrive/pohrl/m2_<model>/` and are **resumable**: if the session disconnects, rerun all cells and it continues.

What to send back: the zip made by the last cell.

In [ ]:
# 1. GPU check: L4 / A100 / G4 recommended (T4 works for M2 with a <=2B model)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install (vLLM brings a matching PyTorch). Takes a few minutes; restart the runtime if Colab asks.
!pip install -q vllm peft datasets sympy antlr4-python3-runtime==4.11

In [ ]:
# 3. Mount Drive and unpack the code. Upload pohrl.zip to MyDrive/pohrl/ first.
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/pohrl'
!mkdir -p /content/code && cd /content/code && unzip -oq {BASE}/pohrl.zip
import sys; sys.path.insert(0, '/content/code/pohrl')
import pohrl, pohrl_llm; print('code ok')

In [ ]:
# 4. Configuration
import torch
MODEL = 'Qwen/Qwen3-1.7B'        # L4/A100: 'Qwen/Qwen3-4B'
DTYPE = 'bfloat16' if torch.cuda.get_device_capability()[0] >= 8 else 'float16'
G = 4                            # samples per problem for PO-HRL and for CoT
N_MATH = 100                     # MATH500 problems (hardest N kept); AIME 2024 is added in full
OUT = f"{BASE}/m2v2_{MODEL.split('/')[-1]}"   # v2: budget forcing + tolerant parsing + dependency check

In [ ]:
# 5. Data (hub ids are configurable in pohrl_llm/data.py; check them on huggingface.co if a load fails)
import random
from pohrl_llm import data as D
math500 = D.load_hf('math500')
random.Random(0).shuffle(math500)
math500 = sorted(math500, key=lambda r: r.get('level', 0))[-N_MATH:] if N_MATH < len(math500) else math500
aime = D.load_hf('aime24')
for r in aime: r['id'] = 'aime24-' + r['id']
problems = math500 + aime
print(len(math500), 'MATH500 +', len(aime), 'AIME24')

In [ ]:
# 6. Model
from pohrl_llm.backends import VLLMBackend
from pohrl_llm import prompts as PR
backend = VLLMBackend(MODEL, dtype=DTYPE, gpu_memory_utilization=0.85, max_model_len=8192)
render = lambda msgs: PR.chat(msgs, backend.tokenizer, enable_thinking=False)

In [ ]:
# 7. Run (resumable). Roughly: a few hours for ~330 problems on an A100 with a 4B model.
from pohrl_llm import run_m2
summary = run_m2.run(problems, backend, render, OUT, G=G, batch=16, temperature=0.7, audit=True)
import json; print(json.dumps(summary, indent=1))

In [ ]:
# 8. Plots: plan width distribution, accuracy by width bucket
import json, matplotlib.pyplot as plt, numpy as np
rows = [json.loads(l) for l in open(f'{OUT}/m2_results.jsonl')]
w = [r['width'] for r in rows if not r['fallback']]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(w, bins=range(1, max(w) + 2), color='#888'); ax[0].set_xlabel('plan width'); ax[0].set_ylabel('problems')
b = summary['by_width']; ks = list(b)
ax[1].bar(np.arange(len(ks)) - 0.2, [b[k]['cot_acc'] for k in ks], 0.4, color='#bbb', label='CoT')
ax[1].bar(np.arange(len(ks)) + 0.2, [b[k]['pohrl_acc'] for k in ks], 0.4, color='#2a6fdb', label='PO-HRL (frozen)')
ax[1].set_xticks(range(len(ks))); ax[1].set_xticklabels(ks); ax[1].set_xlabel('width bucket'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{OUT}/m2_overview.png', dpi=150)

In [ ]:
# 9. Package results to send back
!cd "{OUT}" && zip -qr ../m2_results_$(basename "{OUT}").zip . && ls -la ..